## MarkerDB API

Rest API:
Endpoint: `http://markerdb.ca/api/v1/generalapi/generalrequest`

HTTP Method: GET (retrieve data)

In [61]:
import requests
import pandas as pd
import time

URL = "http://markerdb.ca/api/v1/generalapi/generalrequest"
params = {
    "api_key": "cc69fb7c1a4809bbae7c0701d5188c17",  
    "category": "Diagnostic",
    "biomarker_type": "Chemical",
}

all_records = []
page = 1

print("Starting full API data extraction...")

while True:
    print(f"Fetching page {page}...", end="\r")
    params["page"] = page

    if page > 400:
        break
    
    response = requests.get(URL, params=params)
    if response.status_code != 200:
        print(f"Stopped. Status code: {response.status_code}")
        break
        
    data = response.json()
    biomarkers = data.get("biomarkers", {})
    
    # Natural stop: breaks when the API returns an empty object {}
    if not biomarkers:
        print("No more data found. Finishing extraction.")
        break
        
    for raw_key, condition_list in biomarkers.items():
        for condition in condition_list:
            all_records.append({
                "biomarker_name": condition.get("biomarker_name"),
                "mdbid": condition.get("mdbid"),
                "condition_id": condition.get("condition_id"),
                "condition_name": condition.get("condition_name")
            })
            
    time.sleep(0.01) # Light delay to avoid getting blocked
    page += 1

if all_records:
    df = pd.DataFrame(all_records)
    df.to_csv("markerdb_all_raw_data.csv", index=False)
    print(f"\nDone! Saved {len(df)} total rows to 'markerdb_all_raw_data.csv'")
else:
    print("No data was collected.")

Starting full API data extraction...
Fetching page 401...
Done! Saved 2783 total rows to 'markerdb_all_raw_data.csv'


In [42]:
# biomarkers_dict = data.get("biomarkers", {})

# if not biomarkers_dict:
#     raise Exception("No biomarkers key found in API response")

# rows = []

# for group, items in biomarkers_dict.items():
#     for item in items:
#         rows.append(item)

# df = pd.DataFrame(rows)

# print("Rows:", len(df))
# print("Columns:", df.columns.tolist())

# if df.empty:
#     raise Exception("No data recieved")

# #api data cleaning



Exception: No biomarkers key found in API response

In [62]:
df["condition_name"] = (
    df["condition_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("\nTop conditions in dataset:")
print(df["condition_name"].value_counts().head(15))

mask_diabetes = (
    df["condition_name"].str.contains("diabetes", na=False) |
    df["condition_name"].str.contains("glucose", na=False) |
    df["condition_name"].str.contains("glyc", na=False) |
    df["condition_name"].str.contains("hba1c", na=False)
)

print("\nDiabetes matches:", mask_diabetes.sum())

mask_kidney = (
    df["condition_name"].str.contains("kidney", na=False) |
    df["condition_name"].str.contains("renal", na=False) |
    df["condition_name"].str.contains("ckd", na=False) |
    df["condition_name"].str.contains("nephro", na=False)
)

print("Kidney matches:", mask_kidney.sum())

mask_cardio = (
    df["condition_name"].str.contains("heart", na=False) |
    df["condition_name"].str.contains("cardiac", na=False) |
    df["condition_name"].str.contains("coronary", na=False) |
    df["condition_name"].str.contains("vascular", na=False) |
    df["condition_name"].str.contains("hypertens", na=False) |
    df["condition_name"].str.contains("stroke", na=False) |
    df["condition_name"].str.contains("atheros", na=False)
)

print("Cardio matches:", mask_cardio.sum())

mask = mask_diabetes | mask_kidney | mask_cardio

print("\nTOTAL selected:", mask.sum())

clean_df = df.loc[mask].drop_duplicates()

print("Final cleaned rows:", len(clean_df))


Top conditions in dataset:
condition_name
pregnancy                                       352
obesity                                         260
eosinophilic esophagitis                        149
alzheimer's disease                             108
uremia                                           80
phenylketonuria                                  53
lewy body dementia                               47
chronic kidney disease                           40
meningitis                                       39
diabetes mellitus type 2                         38
schizophrenia                                    36
celiac disease                                   34
preeclampsia/eclampsia                           33
autosomal dominant polycystic kidney disease     28
canavan disease                                  27
Name: count, dtype: int64

Diabetes matches: 68
Kidney matches: 156
Cardio matches: 32

TOTAL selected: 256
Final cleaned rows: 159


In [63]:
clean_df = df.loc[mask].drop_duplicates()

print("Rows after mas filtering:", len(clean_df))

if clean_df.empty:
    raise Exception("clean_df is empty")

print("\nSample cleaned data:")
print(clean_df.head())

Rows after mas filtering: 159

Sample cleaned data:
       biomarker_name        mdbid  condition_id  \
5   1-Methylhistidine  MDB00000001            83   
6   1-Methylhistidine  MDB00000001           203   
11    Androstenedione  MDB00000026             4   
14    Androstenedione  MDB00000026          1085   
15    Androstenedione  MDB00000026          4299   

                                       condition_name  
5                              chronic kidney disease  
6                                      kidney disease  
11                     congenital adrenal hyperplasia  
14              lipoid congenital adrenal hyperplasia  
15  congenital adrenal insufficiency with 46,xy se...  


In [64]:
output_dir = "../final_cleaned_data_files"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "cleaned_markerdb_api.csv")

try:
    clean_df.to_csv(output_file, index=False)
    print("File saved")
    print("Path:", output_file)

except Exception as e:
    print("couldn't save file")


File saved
Path: ../final_cleaned_data_files\cleaned_markerdb_api.csv
